# NYC taxi pickups: one reproducible workflow

This notebook replaces the preparation scripts and nine analysis skeletons. Run **Restart Kernel & Run All**. It cleans the original 20-column trip CSV, aggregates all records in disk-backed DuckDB, and only then loads a top-zone panel and a city series into Python. No raw-trip dataframe, dense zone design matrix, or extra notebook is required.

**Question:** can calendar patterns improve a rolling one-hour-ahead pickup forecast relative to the same clock hour last week? Separately, how is observed weather associated with completed pickups after calendar adjustment? Completed pickups measure served trips, not latent demand or a causal weather effect.

**Split:** Jan–Aug training; Sep–Oct validation; Nov–Dec final test. Zone selection, scaling, target retransformation and coefficients use training only. Validation chooses the forecast; test is evaluated in one final cell. Actual future weather and same-hour trip summaries never enter the forecasting models. This is a rolling forecast, so last week's observed demand is available at each prediction time.

Read [the performance review](../docs/performance_review.md) for changes and limits. Raw data stays local and immutable. Full-data timing must be measured on your machine; synthetic checks are not a 50M-row benchmark.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import gc
import hashlib
import csv
import json
import os
import platform
import sys

import duckdb
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse.linalg import lsqr
import statsmodels.api as sm
import patsy
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
PIPELINE_VERSION = 1
WEATHER = ['temperature_2m', 'precipitation', 'snowfall', 'snow_depth',
           'cloud_cover', 'shortwave_radiation', 'is_day']
TYPES = {
    'VendorID': 'INTEGER', 'tpep_pickup_datetime': 'TIMESTAMP',
    'tpep_dropoff_datetime': 'TIMESTAMP', 'passenger_count': 'DOUBLE',
    'trip_distance': 'DOUBLE', 'RatecodeID': 'DOUBLE', 'store_and_fwd_flag': 'VARCHAR',
    'PULocationID': 'INTEGER', 'DOLocationID': 'INTEGER', 'payment_type': 'INTEGER',
    'fare_amount': 'DOUBLE', 'extra': 'DOUBLE', 'mta_tax': 'DOUBLE',
    'tip_amount': 'DOUBLE', 'tolls_amount': 'DOUBLE', 'improvement_surcharge': 'DOUBLE',
    'total_amount': 'DOUBLE', 'congestion_surcharge': 'DOUBLE',
    'Airport_fee': 'DOUBLE', 'cbd_congestion_fee': 'DOUBLE',
}

@dataclass
class Config:
    root: Path
    memory_limit: str = '2GB'
    threads: int = 4
    top_zones: int = 50
    train_end: str = '2025-09-01'
    val_end: str = '2025-11-01'
    rebuild: bool = False

    @property
    def raw_taxi(self):
        return self.root / 'data/taxi.csv'

    @property
    def raw_weather(self):
        return self.root / 'data/weather.csv'

    @property
    def processed(self):
        return self.root / 'data/processed'

    @property
    def outputs(self):
        return self.root / 'outputs'


def quote(value):
    # SQL string literals, including Windows paths and apostrophes in a directory.
    return "'" + str(value).replace("'", "''") + "'"


def connect(cfg):
    if not 1 <= cfg.top_zones <= 263 or cfg.threads < 1:
        raise ValueError('Use 1..263 zones and a positive thread count.')
    if not pd.Timestamp('2025-01-08') < pd.Timestamp(cfg.train_end) < pd.Timestamp(cfg.val_end) < pd.Timestamp('2026-01-01'):
        raise ValueError('Split boundaries must be ordered inside 2025 after the baseline warmup.')
    cfg.processed.mkdir(parents=True, exist_ok=True)
    spill = cfg.processed / 'spill'
    spill.mkdir(exist_ok=True)
    con = duckdb.connect(str(cfg.processed / 'taxi_work.duckdb'))
    con.execute(f'SET memory_limit = {quote(cfg.memory_limit)}')
    con.execute(f'SET threads = {int(cfg.threads)}')
    con.execute(f'SET temp_directory = {quote(spill.as_posix())}')
    con.execute('SET preserve_insertion_order = false')
    return con

## 1. Configuration and resource budget

Install `requirements.txt` in the kernel environment first. Start with 2 GB for DuckDB and four threads; leave additional RAM for Python, the kernel and operations outside DuckDB's buffer manager. This setting is not a process-wide RAM ceiling. Use a local SSD for `data/processed/`, with space for the staged database and spill files (start with several times the raw CSV size and monitor it). More threads can increase peak memory.

Set `REBUILD=True` after an intentional source replacement or pipeline change. The cache matches source path, size and nanosecond modification time plus the configuration and pipeline version; it does not read and hash a multi-GB input on every restart. Metadata-preserving replacements require a forced rebuild.

The raw files must cover all of 2025. CSV parse errors stop the build instead of silently discarding records. This pipeline does not infer which absent hours reflect missing source coverage: verify source completeness before interpreting zero-filled bins as zero pickups.

In [ ]:
# Works from the repo root or notebooks/. Set NYC_TAXI_ROOT for another location.
root = Path(os.environ.get('NYC_TAXI_ROOT', Path.cwd())).resolve()
if root.name == 'notebooks':
    root = root.parent
cfg = Config(root=root, top_zones=int(os.environ.get('NYC_TAXI_TOP_ZONES', '50')), rebuild=False)
for folder in ['tables', 'figures']:
    (cfg.outputs / folder).mkdir(parents=True, exist_ok=True)
print({'root': str(cfg.root), 'DuckDB_memory': cfg.memory_limit,
       'threads': cfg.threads, 'zones': cfg.top_zones})

## 2. Weather alignment and one-pass trip preparation

Open-Meteo's supplied file uses fixed EST (UTC−5), not changing New York offsets. Shift end-of-hour precipitation, snow and radiation while still on the regular fixed-offset grid, then convert to New York clock labels. Spring 02:00 disappears; the two fall-back 01:00 hours merge. Amounts sum and rates average. The final hour lacks the following weather observation and is excluded from modeling, without imputation.

The trip CSV is read once into a **persistent** stage. Sequential cleaning counts reconcile exactly: raw = kept + removed. Deduplication uses equality of **all 20 original columns** before normalization, rather than a hash alone. It deliberately reads dropped fields to preserve the original duplicate rule. DuckDB performs the large deduplication and aggregation with disk spill; Python receives only a small audit table. Intermediate full-trip Parquet exports and dropoff features are omitted because these analyses do not use them.

In [ ]:
def clean_weather(path):
    wx = pd.read_csv(path, skiprows=3)
    wx.columns = [c.split(' (')[0] for c in wx.columns]
    required = {'time', *WEATHER}
    if not required.issubset(wx.columns):
        raise ValueError(f'Missing weather columns: {sorted(required - set(wx.columns))}')
    original_time = pd.to_datetime(wx['time'], errors='raise')
    expected = pd.date_range('2025-01-01', '2025-12-31 23:00', freq='h')
    if not original_time.reset_index(drop=True).equals(pd.Series(expected)):
        raise ValueError('Weather must be the complete, ordered 8760-hour fixed EST 2025 grid.')
    wx = wx[['time', *WEATHER]].copy()
    wx[WEATHER] = wx[WEATHER].apply(pd.to_numeric, errors='raise')
    if not np.isfinite(wx[WEATHER].to_numpy()).all():
        raise ValueError('Raw weather has missing or nonfinite observations.')
    shifted = ['precipitation', 'snowfall', 'shortwave_radiation']
    wx[shifted] = wx[shifted].shift(-1)
    wx['time'] = (original_time.dt.tz_localize('Etc/GMT+5')
                  .dt.tz_convert('America/New_York').dt.tz_localize(None))
    combine = {c: 'mean' for c in WEATHER}
    combine.update({'precipitation': lambda s: s.sum(min_count=1),
                    'snowfall': lambda s: s.sum(min_count=1), 'is_day': 'max'})
    wx = wx.groupby('time', as_index=False).agg(combine)
    if len(wx) != 8759 or not wx['time'].is_unique:
        raise ValueError('Weather clock-hour grid is not unique/complete.')
    missing = wx[WEATHER].isna().any(axis=1)
    if wx.loc[missing, 'time'].tolist() != [pd.Timestamp('2025-12-31 23:00')]:
        raise ValueError('Unexpected missing weather after alignment.')
    return wx


# First failing rule wins; counts are mutually exclusive rather than overlapping flags.
RULES = [
    ('pickup_outside_2025', "tpep_pickup_datetime IS NULL OR tpep_pickup_datetime < TIMESTAMP '2025-01-01' OR tpep_pickup_datetime >= TIMESTAMP '2026-01-01'"),
    ('unknown_pickup_zone', 'PULocationID IS NULL OR PULocationID NOT BETWEEN 1 AND 263'),
    ('nonexistent_clock_hour', "date_trunc('hour', tpep_pickup_datetime) = TIMESTAMP '2025-03-09 02:00'"),
    ('nonfinite_measure', '(trip_distance IS NOT NULL AND NOT isfinite(trip_distance)) OR (fare_amount IS NOT NULL AND NOT isfinite(fare_amount))'),
    ('non_trip_payment', 'payment_type IN (3, 6) OR (payment_type = 4 AND NOT coalesce(trip_distance > 0 AND fare_amount >= 0, false))'),
    ('negative_fare', 'fare_amount < 0 AND NOT coalesce(payment_type = 0 AND trip_distance > 0, false)'),
    ('dropoff_before_pickup', 'tpep_dropoff_datetime IS NULL OR duration_s IS NULL OR duration_s < 0'),
    ('impossible_duration', 'duration_s = 0 OR duration_s > 21600'),
    ('zero_distance', '(trip_distance IS NULL OR trip_distance <= 0) AND NOT coalesce(duration_s >= 60 AND fare_amount > 0, false)'),
    ('extreme_distance', 'trip_distance > 100'),
    ('impossible_speed', 'duration_s > 0 AND trip_distance / (duration_s / 3600.0) > 80'),
]


def build_trip_aggregates(con, cfg):
    with cfg.raw_taxi.open(encoding='utf-8-sig', newline='') as handle:
        header = next(csv.reader(handle), [])
    if len(header) != 20 or set(header) != set(TYPES):
        raise ValueError('Taxi CSV must contain exactly the 20 original columns; inspect the source schema.')
    types = ', '.join(f'{quote(c)}: {quote(t)}' for c, t in TYPES.items())
    reason = ' '.join(f'WHEN coalesce(({condition}), false) THEN {quote(name)}'
                      for name, condition in RULES)
    con.execute(f"""
        CREATE OR REPLACE TABLE staged AS
        WITH raw AS (
            SELECT *, date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime) AS duration_s
            FROM read_csv({quote(cfg.raw_taxi.as_posix())}, header=true,
                          types={{{types}}}, ignore_errors=false)
        )
        SELECT *, CASE {reason} END AS reason FROM raw
    """)
    # Exactly one scan for the whole removal report, not one scan per rule.
    counts = dict(con.execute('SELECT reason, count(*) FROM staged GROUP BY reason').fetchall())
    raw_rows = int(sum(counts.values()))
    if raw_rows == 0:
        raise ValueError('Taxi source is empty.')
    original_columns = ', '.join('"' + c + '"' for c in TYPES)
    con.execute(f"""
        CREATE OR REPLACE TABLE observed AS
        WITH exact_trips AS (
            SELECT DISTINCT {original_columns} FROM staged WHERE reason IS NULL
        )
        SELECT PULocationID AS zone, date_trunc('hour', tpep_pickup_datetime) AS hour,
               count(*)::BIGINT AS pickups,
               coalesce(sum(fare_amount) FILTER (fare_amount >= 0), 0) AS fare_sum,
               count(*) FILTER (fare_amount >= 0)::BIGINT AS fare_n,
               coalesce(sum(trip_distance) FILTER (trip_distance > 0), 0) AS distance_sum,
               count(*) FILTER (trip_distance > 0)::BIGINT AS distance_n
        FROM exact_trips GROUP BY zone, hour
    """)
    kept = int(con.execute('SELECT coalesce(sum(pickups), 0) FROM observed').fetchall()[0][0])
    removed = {name: int(counts.get(name, 0)) for name, _ in RULES}
    removed['exact_duplicate'] = int(counts.get(None, 0) - kept)
    if raw_rows != kept + sum(removed.values()) or kept == 0:
        raise ValueError('Trip reconciliation failed or no valid trips remain.')
    con.execute('DROP TABLE staged')
    con.execute('CHECKPOINT')
    return {'raw_rows': raw_rows, 'kept_rows': kept,
            'removed_rows': sum(removed.values()), 'rules': removed}


def prepare_panel(con, cfg, weather):
    if len(weather) != 8759 or not weather['time'].is_unique:
        raise ValueError('Expected unique weather for every NYC clock hour.')
    con.register('weather_input', weather)
    con.execute("""
        CREATE OR REPLACE TABLE zone_hour AS
        WITH grid AS (
            SELECT z.zone::SMALLINT AS zone, h.hour
            FROM range(1, 264) z(zone),
                 generate_series(TIMESTAMP '2025-01-01', TIMESTAMP '2025-12-31 23:00', INTERVAL 1 HOUR) h(hour)
            WHERE h.hour <> TIMESTAMP '2025-03-09 02:00'
        )
        SELECT g.*, coalesce(o.pickups, 0)::BIGINT AS pickups,
               coalesce(o.fare_sum, 0) AS fare_sum, coalesce(o.fare_n, 0) AS fare_n,
               coalesce(o.distance_sum, 0) AS distance_sum, coalesce(o.distance_n, 0) AS distance_n,
               w.* EXCLUDE (time)
        FROM grid g LEFT JOIN observed o USING (zone, hour)
        LEFT JOIN weather_input w ON g.hour = w.time
    """)
    n, total = con.execute('SELECT count(*), sum(pickups) FROM zone_hour').fetchall()[0]
    observed_total = con.execute('SELECT sum(pickups) FROM observed').fetchall()[0][0]
    if n != 2_303_617 or total != observed_total:
        raise ValueError('Grid dimensions or pickup conservation failed.')
    # Only the documented final weather hour may be incomplete.
    missing_condition = ' OR '.join(f'{c} IS NULL' for c in WEATHER)
    missing_hours = con.execute(f'SELECT DISTINCT hour FROM zone_hour WHERE {missing_condition} ORDER BY hour').fetchall()
    if missing_hours != [(pd.Timestamp('2025-12-31 23:00').to_pydatetime(),)]:
        raise ValueError(f'Unexpected missing weather hours: {missing_hours[:5]}')
    con.execute(f"""
        CREATE OR REPLACE TABLE calendar AS
        SELECT DISTINCT hour, hour::DATE AS date,
               hour(hour)::SMALLINT AS hod, isodow(hour)::SMALLINT AS dow,
               month(hour)::SMALLINT AS month,
               (isodow(hour) >= 6)::INTEGER AS weekend,
               CASE WHEN isodow(hour) >= 6 THEN hour(hour) ELSE 0 END::SMALLINT AS wknd_hod,
               CASE WHEN hour = TIMESTAMP '2025-11-02 01:00' THEN 2.0 ELSE 1.0 END AS exposure_hours,
               CASE WHEN hour < TIMESTAMP {quote(cfg.train_end)} THEN 'train'
                    WHEN hour < TIMESTAMP {quote(cfg.val_end)} THEN 'val' ELSE 'test' END AS split
        FROM zone_hour
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE top_zones AS
        SELECT zone, sum(pickups)::BIGINT AS training_pickups
        FROM zone_hour WHERE hour < TIMESTAMP {quote(cfg.train_end)}
        GROUP BY zone ORDER BY training_pickups DESC, zone LIMIT {int(cfg.top_zones)}
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE model_sample AS
        SELECT a.zone, a.hour, a.pickups, c.* EXCLUDE (hour),
               {', '.join('a.'+w for w in WEATHER)},
               b.pickups AS base_lag168,
               CASE WHEN b.hour = TIMESTAMP '2025-11-02 01:00' THEN 2.0 ELSE 1.0 END AS lag_exposure
        FROM zone_hour a JOIN top_zones USING (zone) JOIN calendar c USING (hour)
        LEFT JOIN zone_hour b ON a.zone = b.zone AND b.hour = a.hour - INTERVAL 7 DAY
        WHERE a.hour <> TIMESTAMP '2025-12-31 23:00'
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE city_hourly AS
        SELECT c.*, s.city_pickups, {', '.join('w.'+v for v in WEATHER)}
        FROM (SELECT hour, sum(pickups)::BIGINT AS city_pickups FROM zone_hour GROUP BY hour) s
        JOIN calendar c USING (hour) JOIN weather_input w ON s.hour = w.time
        WHERE s.hour <> TIMESTAMP '2025-12-31 23:00'
    """)
    bad = con.execute("SELECT count(*) FROM model_sample WHERE split <> 'train' AND base_lag168 IS NULL").fetchall()[0][0]
    if bad:
        raise ValueError(f'{bad} evaluated rows have no weekly baseline.')
    # Export directly in DuckDB, then rename completed checkpoints atomically.
    for table in ['zone_hour', 'model_sample', 'city_hourly']:
        path = cfg.processed / (table + '.parquet')
        partial = path.with_suffix('.partial.parquet')
        con.execute(f'COPY (SELECT * FROM {table} ORDER BY ' +
                    ('zone, hour' if table != 'city_hourly' else 'hour') +
                    f') TO {quote(partial.as_posix())} (FORMAT PARQUET, COMPRESSION ZSTD)')
        partial.replace(path)
    panel = con.execute('SELECT * FROM model_sample ORDER BY zone, hour').df()
    city = con.execute('SELECT * FROM city_hourly ORDER BY hour').df()
    if len(panel) != cfg.top_zones * 8758 or len(city) != 8758:
        raise ValueError('Model dimensions do not match the configured zones/calendar.')
    return panel, city


def source_signature(cfg):
    sources = []
    for p in [cfg.raw_taxi, cfg.raw_weather]:
        s = p.stat()  # fail clearly even if a stale cache exists
        sources.append({'path': str(p.resolve()), 'size': s.st_size, 'mtime_ns': s.st_mtime_ns})
    return {'version': PIPELINE_VERSION, 'sources': sources, 'top_zones': cfg.top_zones,
            'train_end': cfg.train_end, 'val_end': cfg.val_end,
            'duckdb': duckdb.__version__, 'pandas': pd.__version__}


def run_pipeline(cfg):
    started = perf_counter()
    cfg.outputs.mkdir(parents=True, exist_ok=True)
    fingerprint = source_signature(cfg)
    manifest_path = cfg.processed / 'manifest.json'
    required = [cfg.processed / f'{t}.parquet' for t in ['zone_hour', 'model_sample', 'city_hourly']]
    cached = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    cache_hit = bool(not cfg.rebuild and cached and cached['signature'] == fingerprint and all(p.exists() for p in required))
    con = connect(cfg)
    try:
        if cache_hit:
            panel = con.execute(f'SELECT * FROM read_parquet({quote(required[1].as_posix())}) ORDER BY zone, hour').df()
            city = con.execute(f'SELECT * FROM read_parquet({quote(required[2].as_posix())}) ORDER BY hour').df()
            audit = cached['audit']
        else:
            manifest_path.unlink(missing_ok=True)  # a failed rebuild must never validate mixed outputs
            weather = clean_weather(cfg.raw_weather)  # fail fast before reading the large trip CSV
            audit = build_trip_aggregates(con, cfg)
            panel, city = prepare_panel(con, cfg, weather)
            manifest = {'signature': fingerprint, 'audit': audit}
            partial = manifest_path.with_suffix('.partial.json')
            partial.write_text(json.dumps(manifest, indent=2))
            partial.replace(manifest_path)
    finally:
        con.close()
    # Save machine/environment and pipeline timing; peak process RSS includes all earlier kernel work.
    peak_rss = None
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        peak_rss = rss / (1024**2 if sys.platform == 'darwin' else 1024)
    except ImportError:
        pass  # Windows: record RSS with your system monitor instead.
    benchmark = {'cache_hit': cache_hit, 'pipeline_seconds': perf_counter()-started,
                 'peak_process_rss_MiB': peak_rss, 'python': platform.python_version(),
                 'duckdb': duckdb.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
                 'memory_limit': cfg.memory_limit, 'threads': cfg.threads, **audit}
    (cfg.outputs / 'pipeline_benchmark.json').write_text(json.dumps(benchmark, indent=2))
    return panel, city, audit

In [ ]:
panel, city, audit = run_pipeline(cfg)
display(pd.DataFrame([{'rule': k, 'removed': v} for k,v in audit['rules'].items()]))
print({k:v for k,v in audit.items() if k != 'rules'})
print('Model panel:', panel.shape, 'City hours:', len(city))
print('Panel memory MiB:', round(panel.memory_usage(deep=True).sum()/1024**2, 1))
pd.DataFrame([{'rule': k, 'removed': v} for k,v in audit['rules'].items()]).to_csv(
    cfg.outputs / 'tables/cleaning_report.csv', index=False)

## 3. Sample checks and useful descriptive views

Every zone has the same clock-hour grid; no weather join may multiply rows. The fall-back 01:00 bin represents **two real hours**. Models estimate pickups per exposure hour and multiply predictions by the known exposure; the weekly baseline receives the same adjustment. At the spring boundary, a missing last-week clock hour is not replaced with a row from 167 or 169 hours ago. Only training rows have missing weekly lags.

Duration cleaning still uses differences between naive TLC clock timestamps, so trips crossing a DST change can have ambiguous durations; the raw data cannot identify the fold. See the documented limitation before interpreting those cuts. Zero-filled demand also assumes the raw source is complete.

In [ ]:
for frame, keys in [(panel, ['zone','hour']), (city, ['hour'])]:
    assert not frame.duplicated(keys).any()
    assert frame[WEATHER].notna().all().all()
assert (panel['pickups'] >= 0).all()
assert panel.loc[panel['split'] != 'train', 'base_lag168'].notna().all()
assert panel.loc[panel['split']=='train', 'hour'].max() < panel.loc[panel['split']=='val','hour'].min()
assert panel.loc[panel['split']=='val', 'hour'].max() < panel.loc[panel['split']=='test','hour'].min()
split_table = panel.groupby('split', sort=False).agg(rows=('pickups','size'),
    hours=('hour','nunique'), pickups=('pickups','sum'))
display(split_table)
split_table.to_csv(cfg.outputs / 'tables/split_summary.csv')
training_totals = panel.loc[panel['split']=='train'].groupby('zone')['pickups'].sum().sort_values(ascending=False)
training_totals.to_csv(cfg.outputs / 'tables/top_zones.csv')
share = training_totals.sum() / city.loc[city['split']=='train','city_pickups'].sum()
print(f'Training pickups covered by selected zones: {share:.2%}')
# Descriptive plots use training only, so test patterns cannot steer model choice.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_city = city.loc[city['split']=='train'].copy()
train_city.groupby('hod')['city_pickups'].mean().plot(ax=axes[0], title='Training city pickups by clock hour')
train_city.set_index('hour')['city_pickups'].resample('D').sum().plot(ax=axes[1], title='Training daily pickups')
fig.tight_layout(); fig.savefig(cfg.outputs / 'figures/demand_profile.png', dpi=150); plt.show()

## 4. Sparse zone-level forecast ladder

Annual sine/cosine terms replace month dummies: a chronological training set has no November or December category to estimate. One year still gives weak evidence about annual seasonality. All models use exactly the same training rows with known seven-day lags (a transparent first-week warmup); no validation or test observations are fitted.

The design matrix is sparse CSR. Numeric terms are standardized on training only; categorical treatment levels are fixed and a weekend reference-hour curve avoids the dummy trap. LSQR convergence is checked, and only coefficients and compact statistics are retained. This fits the aggregated panel rather than the original 50M+ trips. We keep classical/robust inferential tests on the small city series below, where dense covariance calculations are practical.

Candidates: M0 constant → M1 hour → M2 weekday → M3 annual harmonics → M4 zone → M5 weekend curve → M6 last-week rate. The same-weekday/time baseline is always included in selection. Smearing for the log retransformation is estimated from training residuals only, so validation errors are measured in pickups, not log units.

In [ ]:
def forecast_design(frame, training):
    n = len(frame)
    matrices = [sparse.csr_matrix(np.ones((n,1)))]
    names = ['Intercept']
    blocks = {'M0': [0]}
    def add_category(column, levels):
        lookup = {v:i for i,v in enumerate(levels[1:])}
        col = frame[column].map(lookup).fillna(-1).to_numpy(dtype=int)
        good = col >= 0
        matrices.append(sparse.csr_matrix((np.ones(good.sum()), (np.flatnonzero(good), col[good])),
                                         shape=(n, len(levels)-1)))
        names.extend([f'{column}[{v}]' for v in levels[1:]])
    def mark(model):
        blocks[model] = list(range(len(names)))
    add_category('hod', list(range(24))); mark('M1')
    add_category('dow', list(range(1,8))); mark('M2')
    day = (pd.to_datetime(frame['hour']).dt.dayofyear.to_numpy()-1)/365.0
    seasonal = np.column_stack([fn(2*np.pi*k*day) for k in [1,2] for fn in [np.sin,np.cos]])
    matrices.append(sparse.csr_matrix(seasonal))
    names.extend(['annual_sin1','annual_cos1','annual_sin2','annual_cos2']); mark('M3')
    add_category('zone', sorted(frame.loc[training,'zone'].unique())); mark('M4')
    add_category('wknd_hod', list(range(24))); mark('M5')
    lag = np.log1p(frame['base_lag168'].to_numpy(dtype=float)/frame['lag_exposure'].to_numpy(dtype=float))
    center, scale = lag[training].mean(), lag[training].std()
    if not np.isfinite(scale) or scale < 1e-12:
        raise ValueError('Training last-week demand has no usable variation.')
    matrices.append(sparse.csr_matrix(((lag-center)/scale)[:,None]))
    names.append('log_lag7_scaled'); mark('M6')
    return sparse.hstack(matrices, format='csr'), names, blocks


def sparse_ols(X, y):
    solution = lsqr(X, y, atol=1e-9, btol=1e-9, iter_lim=3000)
    if solution[1] not in (1,2):
        raise RuntimeError(f'LSQR did not converge: stop={solution[1]}, iterations={solution[2]}')
    beta = solution[0]
    residual = y-X@beta
    return {'beta': beta, 'rss': float(residual@residual),
            'smear': float(np.exp(residual).mean()), 'iterations': solution[2]}


def to_pickups(log_rate, smear, exposure):
    rate = np.maximum(np.exp(log_rate)*smear-1, 0)
    result = rate*np.asarray(exposure)
    if not np.isfinite(result).all():
        raise ValueError('Nonfinite predictions; check model conditioning.')
    return result


def metrics(actual, predicted):
    a, p = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    if len(a) == 0 or not (np.isfinite(a).all() and np.isfinite(p).all()):
        raise ValueError('Metrics require nonempty, finite and aligned observations.')
    error = p-a
    return {'MAE': float(np.abs(error).mean()), 'RMSE': float(np.sqrt(np.mean(error**2))),
            'WAPE': float(np.abs(error).sum()/a.sum()) if a.sum() else np.nan,
            'bias': float(error.mean())}

In [ ]:
# Preserve the full sample for EDA, but use a common complete-lag sample for every candidate.
model_panel = panel.loc[panel['base_lag168'].notna()].reset_index(drop=True)
train_mask = (model_panel['split']=='train').to_numpy()
val_mask = (model_panel['split']=='val').to_numpy()
X, feature_names, blocks = forecast_design(model_panel, train_mask)
y = np.log1p(model_panel['pickups'].to_numpy()/model_panel['exposure_hours'].to_numpy())
models, training_rows = {}, []
fit_started = perf_counter()
for name, columns in blocks.items():
    fit = sparse_ols(X[train_mask][:,columns], y[train_mask])
    models[name] = fit
    training_rows.append({'model': name, 'train_rows': int(train_mask.sum()),
                          'parameters': len(columns), 'train_RSS_log_rate': fit['rss'],
                          'smearing': fit['smear'], 'LSQR_iterations': fit['iterations']})
ladder = pd.DataFrame(training_rows)
ladder.to_csv(cfg.outputs / 'tables/forecast_ladder.csv', index=False)
display(ladder)
print(f'Sparse design: {X.shape}, {X.nnz:,} nonzeros, '
      f'{(X.data.nbytes+X.indices.nbytes+X.indptr.nbytes)/1024**2:.1f} MiB; '
      f'fit seconds: {perf_counter()-fit_started:.1f}')

## 5. Validation selects the forecast

Select by validation MAE in original pickup counts, including the baseline as a candidate. Coefficients and smearing stay fixed at their training estimates. The weekly baseline adjusts for two-hour fall-back exposure. In this rolling one-hour-ahead evaluation, a lag can legitimately come from an earlier validation/test week once it is historical at the prediction time. This is not a fixed-origin two-month forecast.

No MAPE is used because many zone-hours have zero pickups. WAPE is undefined if a group has zero total pickups.

In [ ]:
validation = model_panel.loc[val_mask]
base_val = (validation['base_lag168']/validation['lag_exposure']*validation['exposure_hours']).to_numpy()
validation_scores = [{'model':'weekly_baseline', **metrics(validation['pickups'], base_val)}]
for name, fit in models.items():
    prediction = to_pickups(X[val_mask][:,blocks[name]]@fit['beta'], fit['smear'], validation['exposure_hours'])
    validation_scores.append({'model':name, **metrics(validation['pickups'], prediction)})
validation_scores = pd.DataFrame(validation_scores).sort_values(['MAE','model']).reset_index(drop=True)
selected = validation_scores.loc[0,'model']
validation_scores.to_csv(cfg.outputs / 'tables/validation_scores.csv', index=False)
display(validation_scores)
print('Locked forecast:', selected)

## 6. City-level explanation, diagnostics and robust F-tests

Weather is shared across zones, so repeating it across 50 zones does not create more independent weather observations. Use all 263 zones aggregated to the city, training hours only, to keep inferential matrices small. This answers a city-level association question; it is not the same estimand as a zone fixed-effect coefficient.

The calendar model has hour, weekday, annual harmonics and a weekend curve. Joint weather association uses all seven weather variables on the same rows. Rank is checked before inference; a singular design stops with a clear error instead of silently using arbitrary coefficients. SEs are clustered by date (`use_t=True`); HAC at 24 and 168 clock-observation lags is a sensitivity check. Across spring-forward, neighboring rows are not always exactly an elapsed hour apart. Day clusters do not cover cross-day dependence.

Breusch–Pagan is a descriptive heteroskedasticity check under temporal dependence, not a reliable independent-observation p-value. Residual/QQ/ACF plots help reveal misspecification. Wald block tests use robust covariance. No VIF is computed.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.graphics.tsaplots import plot_acf

explain = train_city.copy()
explain['y_city'] = np.log1p(explain['city_pickups']/explain['exposure_hours'])
day = (pd.to_datetime(explain['hour']).dt.dayofyear-1)/365
for k in [1,2]:
    explain[f'sin{k}'] = np.sin(2*np.pi*k*day)
    explain[f'cos{k}'] = np.cos(2*np.pi*k*day)
calendar_formula = '1 + C(hod) + C(dow) + sin1 + cos1 + sin2 + cos2 + C(wknd_hod)'
Z_df = patsy.dmatrix(calendar_formula, explain, return_type='dataframe', NA_action='raise')
Z = Z_df.to_numpy()
weather_raw = explain[WEATHER].to_numpy(dtype=float)
weather_center, weather_scale = weather_raw.mean(axis=0), weather_raw.std(axis=0)
if (weather_scale < 1e-12).any():
    raise ValueError('A training weather column is constant; revise the declared weather block before inference.')
W = (weather_raw-weather_center)/weather_scale
full_X = np.column_stack([Z,W])
if np.linalg.matrix_rank(full_X) != full_X.shape[1]:
    raise ValueError('City explanatory design is rank deficient. Inspect auxiliary regressions; do not report unidentified effects.')
y_city = explain['y_city'].to_numpy()
classic = sm.OLS(y_city, full_X).fit()
groups = pd.factorize(explain['date'])[0]
clustered = classic.get_robustcov_results(cov_type='cluster', groups=groups, use_t=True)
hac24 = classic.get_robustcov_results(cov_type='HAC', maxlags=24, use_t=True)
hac168 = classic.get_robustcov_results(cov_type='HAC', maxlags=168, use_t=True)
# Coefficients/CI in the original weather units; inference from the full fit.
ci = clustered.conf_int()[-len(WEATHER):]/weather_scale[:,None]
effects = pd.DataFrame({'weather': WEATHER, 'coefficient': classic.params[-len(WEATHER):]/weather_scale,
    'SE_day_cluster': clustered.bse[-len(WEATHER):]/weather_scale,
    'SE_HAC24': hac24.bse[-len(WEATHER):]/weather_scale,
    'SE_HAC168': hac168.bse[-len(WEATHER):]/weather_scale,
    'p_day_cluster': clustered.pvalues[-len(WEATHER):], 'CI_low': ci[:,0], 'CI_high': ci[:,1]})
effects['pct_pickups_plus_one_per_hour'] = 100*np.expm1(effects['coefficient'])
effects.to_csv(cfg.outputs / 'tables/weather_associations.csv', index=False)
display(effects)
# Test the weather block and the weekend curve with the same robust covariance and rows.
block_tests = []
for label, indexes in [('weather', list(range(Z.shape[1], full_X.shape[1]))),
                        ('weekend_curve', [i for i,c in enumerate(Z_df.columns) if c.startswith('C(wknd_hod)')])]:
    R = np.eye(full_X.shape[1])[indexes]
    test = clustered.f_test(R)
    block_tests.append({'block':label, 'F':float(np.asarray(test.fvalue).item()),
                        'p':float(np.asarray(test.pvalue).item()),
                        'df_num':float(test.df_num), 'df_denom':float(test.df_denom)})
pd.DataFrame(block_tests).to_csv(cfg.outputs / 'tables/robust_f_tests.csv', index=False)
display(pd.DataFrame(block_tests))
bp = het_breuschpagan(classic.resid, full_X)
print({'BP_LM_descriptive': bp[0], 'BP_p_independence_assumption': bp[1],
       'training_city_hours':len(explain), 'date_clusters':len(np.unique(groups)),
       'snow_hours':int((explain['snowfall']>0).sum())})
fig, axes = plt.subplots(1,3,figsize=(15,4))
axes[0].scatter(classic.fittedvalues, classic.resid, s=3, alpha=.3)
axes[0].set(xlabel='Fitted log rate', ylabel='Residual', title='Training city residuals')
sm.qqplot(classic.resid, line='s', ax=axes[1])
plot_acf(classic.resid, lags=168, ax=axes[2], zero=False)
fig.tight_layout(); fig.savefig(cfg.outputs / 'figures/city_diagnostics.png', dpi=150); plt.show()

## 7. Collinearity through auxiliary regressions; exact joint FWL

Auxiliary regressions project each standardized weather column on the calendar and the other weather columns; high R² and residual near-zero variance flag weak identification. Their conventional F statistics are descriptive, since errors are temporally dependent. Strong pairwise correlation alone does not justify dropping a variable automatically.

For FWL, residualize **both** demand and the entire weather matrix against the same joint calendar design, then regress residual demand on all residualized weather columns together. A scatter plot against one column is only illustrative; with several weather predictors its simple slope is not generally the joint coefficient. The full model supplies robust confidence intervals. These associations describe served pickups per hour plus one, not causal effects or available unmet demand.

In [ ]:
auxiliary = []
for j, variable in enumerate(WEATHER):
    other = np.column_stack([Z, np.delete(W,j,axis=1)])
    fit = sm.OLS(W[:,j],other).fit()
    auxiliary.append({'weather':variable, 'aux_R2':fit.rsquared,
                      'aux_F_descriptive':float(fit.fvalue), 'residual_SD':float(fit.resid.std())})
auxiliary = pd.DataFrame(auxiliary)
auxiliary.to_csv(cfg.outputs / 'tables/weather_collinearity.csv', index=False)
display(auxiliary)
pd.DataFrame(weather_raw, columns=WEATHER).corr().to_csv(cfg.outputs / 'tables/weather_correlations.csv')
# Joint orthogonal projection avoids sequential-demeaning errors on an unbalanced calendar.
qz, _ = np.linalg.qr(Z, mode='reduced')
y_partial = y_city-qz@(qz.T@y_city)
w_partial = W-qz@(qz.T@W)
fwl_beta = np.linalg.lstsq(w_partial, y_partial, rcond=None)[0]
np.testing.assert_allclose(fwl_beta, classic.params[-len(WEATHER):], rtol=1e-7, atol=1e-9)
fwl_check = pd.DataFrame({'weather':WEATHER, 'joint_FWL_original_units':fwl_beta/weather_scale,
    'full_model_original_units':classic.params[-len(WEATHER):]/weather_scale})
fwl_check.to_csv(cfg.outputs / 'tables/fwl_check.csv', index=False)
display(fwl_check)
# Partial out calendar AND other weather columns for an actual one-variable partial regression plot.
j = WEATHER.index('precipitation')
controls = np.column_stack([Z,np.delete(W,j,axis=1)])
qc, _ = np.linalg.qr(controls, mode='reduced')
x_r = W[:,j]-qc@(qc.T@W[:,j]); y_r = y_city-qc@(qc.T@y_city)
fig, ax = plt.subplots(figsize=(6,4)); ax.scatter(x_r,y_r,s=4,alpha=.25)
xx = np.array([x_r.min(),x_r.max()]); ax.plot(xx,classic.params[-len(WEATHER)+j]*xx,color='red')
ax.set(xlabel='Residual standardized precipitation',ylabel='Residual log pickup rate',title='Joint weather partial regression')
fig.tight_layout(); fig.savefig(cfg.outputs / 'figures/precipitation_partial.png',dpi=150); plt.show()

## 8. Final test: selected model versus the weekly baseline

Run this after validation has locked the model. Do not use these results to change the candidates or thresholds and then present a rerun as an untouched test. Coefficients are not refit on validation here, so the selection protocol stays transparent. Skill is 1 − model MAE / baseline MAE; negative skill means the weekly rule is better.

Per-zone and per-hour errors support practical dispatch discussion. Test-period observed weather is deliberately absent from the forecast. Holidays, supply limits, congestion pricing and a single year of seasonality remain limitations; this notebook does not establish a production forecasting service.

In [ ]:
test_mask = (model_panel['split']=='test').to_numpy()
test_rows = model_panel.loc[test_mask].copy()
baseline = (test_rows['base_lag168']/test_rows['lag_exposure']*test_rows['exposure_hours']).to_numpy()
if selected == 'weekly_baseline':
    final_prediction = baseline.copy()
else:
    locked = models[selected]
    final_prediction = to_pickups(X[test_mask][:,blocks[selected]]@locked['beta'], locked['smear'], test_rows['exposure_hours'])
final_scores = pd.DataFrame([
    {'model':'weekly_baseline', **metrics(test_rows['pickups'],baseline)},
    {'model':selected, **metrics(test_rows['pickups'],final_prediction)},
]).drop_duplicates('model')
final_scores.to_csv(cfg.outputs / 'tables/final_test_scores.csv',index=False)
display(final_scores)
base_mae = metrics(test_rows['pickups'],baseline)['MAE']
skill = 1-metrics(test_rows['pickups'],final_prediction)['MAE']/base_mae if base_mae else np.nan
print(f'Locked {selected}: test MAE skill versus weekly baseline = {skill:.2%}')
test_rows['prediction'] = final_prediction; test_rows['baseline'] = baseline
for grouping in ['zone','hod']:
    rows=[]
    for key, group in test_rows.groupby(grouping):
        rows.append({grouping:int(key), **metrics(group['pickups'],group['prediction']),
                     'baseline_MAE':metrics(group['pickups'],group['baseline'])['MAE']})
    pd.DataFrame(rows).to_csv(cfg.outputs / f'tables/test_errors_by_{grouping}.csv',index=False)
# Only the aggregate sample, never 50M raw trips, is exported as predictions.
test_rows[['zone','hour','pickups','prediction','baseline']].to_csv(cfg.outputs / 'tables/test_predictions.csv',index=False)
fig, ax = plt.subplots(figsize=(7,4))
daily = test_rows.assign(date=pd.to_datetime(test_rows['hour']).dt.date).groupby('date')[['pickups','prediction','baseline']].sum()
daily.plot(ax=ax,title='Held-out daily pickups: selected zones')
fig.tight_layout(); fig.savefig(cfg.outputs / 'figures/test_daily_pickups.png',dpi=150); plt.show()
summary = {'selected_model':selected, 'test_skill_MAE':skill, 'training_zone_coverage':float(share),
           'raw_rows':audit['raw_rows'], 'kept_rows':audit['kept_rows'],
           'test_scores':final_scores.to_dict('records'),
           'max_FWL_coefficient_difference':float(np.max(np.abs(fwl_beta-classic.params[-len(WEATHER):])))}
(cfg.outputs / 'analysis_summary.json').write_text(json.dumps(summary,indent=2))
print('Saved reproducible tables/figures and analysis_summary.json under',cfg.outputs)

## Findings to write after running on the team's data

Use generated `analysis_summary.json`, `pipeline_benchmark.json`, and `outputs/tables/` as the evidence. Report the measured raw count and cleaning reconciliation; selected-zone training coverage; the locked model's validation and final test MAE/RMSE/WAPE versus the weekly rule; the worst zones/hours; and weather associations with day-clustered and HAC uncertainty. Report an improvement only if the held-out metrics support it.

For performance reporting, distinguish a cold rebuild from a cache hit, include the machine/RAM/SSD and resource configuration, and measure end-to-end runtime as well as the recorded pipeline time. Peak RSS is the process lifetime high-water mark, includes earlier cells, and is unavailable through `resource` on Windows. Python kernel memory is separate from DuckDB's buffer budget. The closed scratch database can be deleted between runs to reclaim disk; keep Parquet checkpoints plus the manifest to reuse the cache. Exact full-row deduplication remains the main high-cardinality operation and must be benchmarked at actual scale.

References: [DuckDB workload tuning](https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads), [DuckDB out-of-memory guidance](https://duckdb.org/docs/stable/guides/troubleshooting/oom_errors), [SciPy LSQR](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.lsqr.html), [statsmodels robust covariance](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html).